# 03. Paoay: features and crop predictions

Apply the saved three-crop model to matching satellite inputs. Preserve all raw scores before any screening. No training, retrieval or field-label assignment.


## 1. Setup

Finish combined 08 first. Use the unique saved model or set an explicit manifest. Clear current stage selections before starting so incomplete inputs cannot select earlier predictions.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import features, predictions, land_cover, dynamic_world, applicability, season_shift
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

model_manifest = local_settings.get("paoay_model_manifest")  # Set to the fixed_full manifest saved by Training 02.
report_folder = local_path(local_settings.get("report_root", "./data/pipeline_work/inventory"))
paoay_root = report_folder.parent / "paoay"
guidance_pointer = paoay_root / "land_cover/land_cover_stage.json"
feature_root = paoay_root / "features"
feature_pointer = feature_root / "feature_stage.json"
prediction_root = paoay_root / "predictions"
prediction_pointer = prediction_root / "prediction_stage.json"
dynamic_world_pointer = paoay_root / "dynamic_world/dynamic_world_stage.json"
imagery_pointer = paoay_root / "sentinel2/imagery_stage.json"
applicability_root = paoay_root / "applicability"
applicability_pointer = applicability_root / "applicability_stage.json"
season_root = paoay_root / "season_shift"
season_pointer = season_root / "season_shift_stage.json"
chunk_size = 4096
show_preview = True
for pointer in (guidance_pointer, feature_pointer, prediction_pointer, applicability_pointer, season_pointer):
    write_stage_pointer(pointer, status="pending", reason="Notebook 09 model application is incomplete")
stages = ["Verify model and satellite inputs", "Calculate exact model inputs", "Review feature readiness", "Apply fixed model", "Check model applicability", "Compare seasonal curves", "Verify outputs", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook("09_paoay_model_application", stages, report_folder.parent / "progress", {"model": str(model_manifest)})


## 2. Verify model and inputs

Retain the existing guidance dependency for cache compatibility; it is provenance only and never excludes feature pixels. The scientific screening review happens in notebook 04. Exact feature order, formula recipe, dates and quality requirements remain unchanged.


In [ ]:
pipeline_progress.begin_stage("Verify model and satellite inputs")
model_path = features.find_model(report_folder.parent / "analysis", local_path(model_manifest) if model_manifest else None)
imagery_handoff = resolve_stage_pointer(imagery_pointer)
dynamic_world_handoff = resolve_stage_pointer(dynamic_world_pointer)
world_bundle = dynamic_world.load_dynamic_world_bundle(dynamic_world_handoff)
if Path(world_bundle["manifest"]["imagery_handoff"]).resolve() != Path(imagery_handoff).resolve():
    raise ValueError("Satellite sources do not match; finish combined notebook 08")
# Compatibility provenance only: retain existing feature identities and cache reuse.
# No guidance class filters any crop-model input.
guidance_request = land_cover.prepare_request(dynamic_world_handoff, paoay_root / "land_cover", land_cover.DEFAULT_RULES)
guidance_handoff = land_cover.build(guidance_request)
request = features.prepare_request(guidance_handoff, model_path, feature_root)
recipe = features.load_model_recipe(model_path)["recipe"]
print("Selected fixed model:", model_path)
print("Feature design:", recipe["selected_design"], "— ordered predictors:", len(recipe["feature_columns"]))
print("Minimum per-composite clear coverage:", recipe["quality"]["minimum_valid_fraction"])
print("Land-cover filtering: no. Model refitting: no.")
pipeline_progress.end_stage()


## 3. Calculate features

Reuse verified bundles when possible. Otherwise calculate the saved recipe with one pixel-progress bar. All eight bins must pass the original coverage and finite-value checks.


In [ ]:
pipeline_progress.begin_stage("Calculate exact model inputs")
def pixel_progress(items):
    return pipeline_progress.track(items, "Calculate crop-model features", unit="pixel")
feature_handoff = features.build(request, iterator=pixel_progress)
bundle = features.load_feature_bundle(feature_handoff, require_ready=False)
pipeline_progress.end_stage()


## 4. Check readiness

Stop on zero usable rows; keep audits and leave stage selections pending. No imputation or relaxed quality thresholds.


In [ ]:
pipeline_progress.begin_stage("Review feature readiness")
show_table(bundle["coverage"], title="Composite coverage inside the ROI")
show_table(bundle["audit"], title="Pixel readiness and exclusions")
print("ROI pixels:", bundle["manifest"]["roi_pixels"])
print("Ready feature rows:", bundle["manifest"]["row_count"])
print("Matrix shape:", bundle["matrix"].shape)
print("Land-cover filtering applied: no")
if bundle["manifest"]["status"] != "prepared":
    raise RuntimeError("No ready feature pixels; review the coverage and pixel audit above. All current stage selections remain pending.")
pipeline_progress.end_stage()


## 5. Predict

Apply the fixed classifier in computational batches. Other crop species remain possible within the boundaries; confidence cannot reliably detect them.


In [ ]:
pipeline_progress.begin_stage("Apply fixed model")
def inference_progress(items):
    return pipeline_progress.track(items, "Apply saved crop model", unit="batch")
prediction_request = predictions.prepare_request(feature_handoff, prediction_root)
prediction_handoff = predictions.build(prediction_request, chunk_size=chunk_size, iterator=inference_progress)
bundle = predictions.load_prediction_bundle(prediction_handoff)
pipeline_progress.end_stage()


## 6. Check model applicability

The model was trained on European parcels. This step asks, pixel by pixel, whether the study area resembles those training pixels, following Meyer and Pebesma (2021), *Predicting into unknown space?*

- **Dissimilarity index (DI):** distance from a pixel to its nearest training pixel in the 72-predictor space, with each predictor standardized and weighted by its importance in the model, divided by the average distance between training pixels. 0 means identical to a training pixel.
- **Threshold:** the largest DI among the training pixels themselves once outliers are set aside, where each training pixel is compared with other parcels only. A pixel above it is **outside the area of applicability**: the model is extrapolating there and its validation score says nothing about it.
- **How to read it:** a low share inside means most of the map is extrapolation. A high share inside means the pixels look like training pixels, which is necessary but not sufficient: a different crop, or the same crop at a different growth stage, can look like a training pixel.

This is a flag. No prediction is masked, relabelled or removed. The check needs the development feature rows that trained the model; if they are not beside the model package, the step says so and the applicability pointer stays pending.

In [ ]:
pipeline_progress.begin_stage("Check model applicability")
applicability_handoff = None
training_rows, cohort, unavailable = applicability.locate_training_rows(model_path)
if training_rows is None:
    print("Applicability check not run:", unavailable + ".")
    print("Predictions are unaffected; the applicability pointer stays pending.")
else:
    applicability_request = applicability.prepare_request(feature_handoff, applicability_root, training_rows)
    applicability_handoff = applicability.build(applicability_request)
    applicability_bundle = applicability.load_bundle(applicability_handoff)
    show_table([{**row, "count": int(row["count"]),
                 "median_di": f"{float(row['median_di']):.2f}" if row["median_di"] else "",
                 "inside_fraction": f"{float(row['inside_fraction']):.0%}" if row["inside_fraction"] else ""}
                for row in applicability_bundle["summary"]], title="Dissimilarity to the training pixels")
    show_table([{**row, **{key: f"{row[key]:.0%}" if key == "inside_fraction" else f"{row[key]:.2f}"
                           for key in ("inside_fraction", "median_top_probability_inside", "median_top_probability_outside") if row[key] != ""}}
                for row in applicability.by_prediction(applicability_bundle, bundle)], title="Raw predictions by applicability (nothing filtered)")
    details = applicability_bundle["manifest"]
    print(f"Training data: {details['training_pixels']} pixels from {details['training_parcel_groups']} parcels ({cohort}).")
    print(f"Applicability threshold (DI): {details['threshold']:.2f}")
    print(f"Study-area pixels inside the area of applicability: {details['study_inside_fraction']:.0%}")
    print("Inside means similar to training pixels, not correctly classified.")
    if show_preview:
        from IPython.display import display
        import matplotlib.pyplot as plt
        figure = applicability.plot(applicability_bundle)
        preview = applicability_root / "previews" / (details["bundle_id"] + "_applicability.png")
        preview.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(preview, dpi=150)
        display(figure)
        plt.close(figure)
        print("Preview:", preview)
pipeline_progress.end_stage()

## 7. Compare seasonal curves

Applicability asks whether a pixel resembles *any* training pixel. This step asks the sharper question: in these calendar months, do the study area's fields behave like the training fields of each crop? Differences in seasonal timing can reduce cross-region transfer ([Nyborg et al., 2022, TimeMatch](https://arxiv.org/abs/2111.02682)).

- **Blue:** the training pixels of one crop, median per half-month with the middle 50% shaded.
- **Green:** pixels inside the reference areas. **Grey dotted:** every study-area pixel. **Red dashed:** pixels the model labelled as that crop.
- **Table:** peak half-month, amplitude (highest minus lowest median) and mean of each curve, and the shift in half-month bins that best lines up the two curves. A positive shift means the study area runs later than training.

How to read it: curves at a similar level with a small shift support using the same calendar window. A large gap in level or amplitude means the fields are at a different growth stage in these months, so the model compares a growing local crop with a different stage of the training crop. A shift reported "at the test limit" means the true shift may be larger than one four-month window can show.

The study-area groups are not labelled single-crop fields: reference areas hold rotations, and pixels grouped by prediction resemble their class by construction. Nothing is adapted, moved or relabelled.

In [ ]:
pipeline_progress.begin_stage("Compare seasonal curves")
season_handoff = None
if training_rows is None:
    print("Season comparison not run:", unavailable + ".")
elif not season_shift.available(recipe["feature_columns"]):
    print("Season comparison not run: this feature design has no per-bin index predictors.")
else:
    season_request = season_shift.prepare_request(prediction_handoff, season_root, training_rows)
    season_handoff = season_shift.build(season_request)
    season_bundle = season_shift.load_bundle(season_handoff)
    show_table(season_shift.comparison_rows(season_bundle, "NDVI"), title="NDVI: training crop curves against the study area")
    print("Lag convention:", season_bundle["manifest"]["lag_convention"] + ".")
    print("All four indices and every group:", season_bundle["folder"] / "comparison.csv")
    if show_preview:
        from IPython.display import display
        import matplotlib.pyplot as plt
        figure = season_shift.plot(season_bundle, "NDVI")
        preview = season_root / "previews" / (season_bundle["manifest"]["bundle_id"] + "_ndvi.png")
        preview.parent.mkdir(parents=True, exist_ok=True)
        figure.savefig(preview, dpi=150)
        display(figure)
        plt.close(figure)
        print("Preview:", preview)
pipeline_progress.end_stage()

## 8. Verify and publish

Check matching dependencies and saved scores, then publish the existing handoff formats. Areas below describe raw model choices, not verified planted area.


In [ ]:
pipeline_progress.begin_stage("Verify outputs")
show_table([{**row, "code": int(row["code"]), "pixels": int(row["pixels"]), "area_ha": float(row["area_ha"]),
             "percent_of_predicted_pixels": float(row["percent_of_predicted_pixels"])} for row in bundle["summary"]],
           title="Crop-only predicted areas")
manifest = bundle["manifest"]
print("ROI pixels:", manifest["roi_pixels"])
print("Predicted pixels:", manifest["row_count"])
print("ROI pixels without valid crop inputs:", manifest["roi_pixels"] - manifest["row_count"])
print("Tied top probabilities:", manifest["tied_pixels"])
print("Classifier class order:", ", ".join(manifest["classifier_class_order"]))
print("Class codes:", "; ".join(f"{name} = {code}" for name, code in manifest["class_codes"].items()))
features.load_feature_bundle(feature_handoff)
predictions.load_prediction_bundle(prediction_handoff)
write_stage_pointer(guidance_pointer, guidance_handoff)
write_stage_pointer(feature_pointer, feature_handoff)
write_stage_pointer(prediction_pointer, prediction_handoff)
if applicability_handoff is not None:
    applicability.load_bundle(applicability_handoff)
    write_stage_pointer(applicability_pointer, applicability_handoff)
if season_handoff is not None:
    season_shift.load_bundle(season_handoff)
    write_stage_pointer(season_pointer, season_handoff)
pipeline_progress.end_stage()


## 9. Next step

Unknown and screening flags belong to 04. Alternative other-crop training methods remain TODOs.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified features:", feature_handoff)
print("Verified raw predictions:", prediction_handoff)
print("Model applicability:", applicability_handoff or "not run")
print("Season comparison:", season_handoff or "not run")
print("These are forced choices among three crops, including on other crop species and noncrop surfaces.")
print("Next: notebook 04 creates boundary-limited maps, Unknown and separate screening flags.")
print("Model, training partitions and imagery preserved; no fitting or downloads.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()


## Evidence and interpretation

Maps and seasonal diagnostics require dated local reference evidence. Screening and clustering do not establish crop accuracy. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.
